In [1]:
%load_ext autoreload
%autoreload 2
import os
import argparse

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
import warnings

warnings.filterwarnings("ignore")
# We import all our dependencies.
import numpy as np
import torch
from torch.utils.data import DataLoader
from models.lvae import LadderVAE
from boilerplate.dataloader import BCSSDataset, ModeAwareBalancedAnchorBatchSampler, flex_collate
import training
from tqdm import tqdm
import tifffile as tiff
from PIL import Image
import matplotlib.pyplot as plt
from glob import glob

In [2]:

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

data_dir = "/home/sheida.rahnamai/BCSS/"
img_paths = sorted(glob(data_dir+'images_vahadane/*.png'))
lbl_paths = sorted(glob(data_dir+'masks/*.png'))
imgs = {k: np.array(Image.open(path)) for k, path in enumerate(img_paths)}
lbls = {k: np.array(Image.open(path)) for k, path in enumerate(lbl_paths)}

In [3]:
use_wandb = True

patch_size = 64

gaussian_noise_std = None

model_name = "segmentation"
directory_path = "/group/jug/Sheida/HVAE/segmentation/test/"

noiseModel = None

# Training-specific
batch_size = 1024
lr = 3e-5
max_epochs = 1000
num_latents = 3
z_dims = [32] * int(num_latents)
blocks_per_layer = 5
batchnorm = False
free_bits = 0.0

alpha = 1  # weight of the inpainting loss
beta = 1e-2  # weight of the KL loss
gamma = 1  # weight of the contrastive loss

initial_mask_size = 1
final_mask_size = 1
initial_label_size =1
final_label_size = 1
step_interval = 100

contrastive_learning = True
margin = 50  # distance for negative pairs in contrastive learning
lambda_contrastive = 0.5  # weight of the positive pairs in contrastive learning
# (1-lambda_contrastive is the weight of the negative pairs)

mode = 'supervised' #or 'semisupervised' or 'unsupervised'
labeled_ratio = 1  # ratio of labeled data in semisupervised mode
stochastic_block_type = 'mixture'  # 'normal' or 'mixture'
conditional = True  # True for conditional LVAE (conditioned on gt label)
condition_type = 'mlp'  # 'mlp' or 'transformer'
assert (conditional == True and condition_type != None) or conditional == False
n_components = 18  # number of components for prior
n_classes = 18  # number of classes in the dataset
# train data
keys = list(range(105))
train_idx, val_idx = [], []
np.random.seed(42)
np.random.shuffle(keys)
split_idx = int(0.85 * 105)
for k in keys:
    if k < split_idx:
        train_idx.append(k)
    else:
        val_idx.append(k)

# compute mean and std of the data
all_elements = np.concatenate([imgs[key].flatten() for key in train_idx])
data_mean = np.mean(all_elements)
data_std = np.std(all_elements.astype(np.float32))

sample_ratio = 20

# normalizing the data
for key in tqdm(keys, "Normalizing data"):
    imgs[key] = (imgs[key] - data_mean) / data_std

Normalizing data: 100%|██████████| 105/105 [00:07<00:00, 14.73it/s]


In [4]:
all_labels = set()
for k in train_idx:
    uniq = np.unique(lbls[k])
    all_labels.update(uniq.tolist())

print("All unique labels across train dataset:", sorted(all_labels))
print("Total classes:", len(all_labels))

all_labels = set()
for k in val_idx:
    uniq = np.unique(lbls[k])
    all_labels.update(uniq.tolist())

print("All unique labels across val dataset:", sorted(all_labels))
print("Total classes:", len(all_labels))



All unique labels across train dataset: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 15, 17, 18, 19, 20]
Total classes: 17
All unique labels across val dataset: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 18]
Total classes: 16


In [9]:
train_set = BCSSDataset(
    images=imgs,
    labels=lbls,
    patch_size=patch_size,
    label_size=initial_label_size,
    mode=mode,
    n_classes=21,
    ignore_lbl=0,
    ratio=labeled_ratio,
    indices=train_idx,
)

val_set = BCSSDataset(
    images=imgs,
    labels=lbls,
    patch_size=patch_size,
    label_size=initial_label_size,
    mode='supervised',
    n_classes=21,
    ignore_lbl=0,
    ratio=labeled_ratio,
    indices=val_idx,
)


  Class 1 (anchors): 87 samples
  Class 2 (anchors): 84 samples
  Class 3 (anchors): 68 samples
  Class 4 (anchors): 53 samples
  Class 5 (anchors): 10 samples
  Class 6 (anchors): 31 samples
  Class 7 (anchors): 53 samples
  Class 9 (anchors): 22 samples
  Class 10 (anchors): 14 samples
  Class 11 (anchors): 8 samples
  Class 13 (anchors): 16 samples
  Class 15 (anchors): 9 samples
  Class 17 (anchors): 1 samples
  Class 18 (anchors): 56 samples
  Class 19 (anchors): 1 samples
  Class 20 (anchors): 1 samples
  Class 1 (neighbors): 621 samples
  Class 2 (neighbors): 672 samples
  Class 3 (neighbors): 473 samples
  Class 4 (neighbors): 379 samples
  Class 5 (neighbors): 47 samples
  Class 6 (neighbors): 200 samples
  Class 7 (neighbors): 336 samples
  Class 9 (neighbors): 151 samples
  Class 10 (neighbors): 96 samples
  Class 11 (neighbors): 45 samples
  Class 13 (neighbors): 115 samples
  Class 15 (neighbors): 57 samples
  Class 17 (neighbors): 7 samples
  Class 18 (neighbors): 382 sam

In [10]:

train_loader = DataLoader(
    train_set,
    batch_sampler=ModeAwareBalancedAnchorBatchSampler(
        train_set, total_patches_per_batch=batch_size, shuffle=True
    ),
    collate_fn=flex_collate,
)
val_loader = DataLoader(
    val_set,
    batch_sampler=ModeAwareBalancedAnchorBatchSampler(
        val_set, total_patches_per_batch=batch_size, shuffle=False
    ),
    collate_fn=flex_collate,
)


img_shape = (64, 64)

if False:
    model = torch.load(checkpoint, weights_only=False)
    model.update_mode("semisupervised")

else:
    model = LadderVAE(
        z_dims=z_dims,
        blocks_per_layer=blocks_per_layer,
        data_mean=data_mean,
        data_std=data_std,
        noiseModel=noiseModel,
        conv_mult=2,
        device=device,
        batchnorm=batchnorm,
        free_bits=free_bits,
        img_shape=img_shape,
        grad_checkpoint=True,
        mask_size=initial_mask_size,
        contrastive_learning=contrastive_learning,
        margin=margin,
        lambda_contrastive=lambda_contrastive,
        stochastic_block_type=stochastic_block_type,
        conditional=conditional,
        condition_type=condition_type,
        n_components=n_components,
        training_mode=mode,
        labeled_ratio=labeled_ratio,
    ).cuda()
print(model)
model.train()  # Model set in training mode

training.train_network(
    model=model,
    lr=lr,
    max_epochs=max_epochs,
    directory_path=directory_path,
    batch_size=batch_size,
    alpha=alpha,
    beta=beta,
    gamma=gamma,
    train_loader=train_loader,
    val_loader=val_loader,
    gaussian_noise_std=gaussian_noise_std,
    model_name=model_name,
    gradient_scale=256,
    use_wandb=use_wandb,
    max_grad_norm=1,
    initial_label_size=initial_label_size,
    final_label_size=final_label_size,
    initial_mask_size=initial_mask_size,
    final_mask_size=final_mask_size,
    step_interval=step_interval,
)


LadderVAE(
  (first_bottom_up): Sequential(
    (0): Conv2d(1, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BlurPool()
    (2): ELU(alpha=1.0)
    (3): BottomUpDeterministicResBlock(
      (res): ResidualBlock(
        (block): Sequential(
          (0): ELU(alpha=1.0)
          (1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (2): ELU(alpha=1.0)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        )
      )
    )
  )
  (top_down_layers): ModuleList(
    (0-1): 2 x TopDownLayer(
      (deterministic_block): Sequential(
        (0): TopDownDeterministicResBlock(
          (pre_conv): ConvTranspose2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1))
          (res): ResidualBlock(
            (block): Sequential(
              (0): ELU(alpha=1.0)
              (1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (2): ELU(alpha=1.0)
       

wandb: Currently logged in as: sheida-rk (juglab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting epoch 0


Training: 0it [00:00, ?it/s]


KeyError: 'name'